### Визуализация COVID ###

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

covid_data = pd.read_csv('data/Visualization/covid_data.csv')
display(Markdown("### covid_data"))
display(covid_data.head())

covid_data

vaccinations_data = pd.read_csv('data/Visualization/country_vaccinations.csv')
vaccinations_data = vaccinations_data[
    ['country', 'date', 'total_vaccinations', 
     'people_vaccinated', 'people_vaccinated_per_hundred',
     'people_fully_vaccinated', 'people_fully_vaccinated_per_hundred',
     'daily_vaccinations', 'vaccines']
]
display(Markdown("### vaccinations_data"))
display(vaccinations_data.head())

covid_data = covid_data.groupby(
    ['date', 'country'], 
    as_index=False
)[['confirmed', 'deaths', 'recovered']].sum()

covid_data['date'] = pd.to_datetime(covid_data['date'])

covid_data['active'] = covid_data['confirmed'] - covid_data['deaths'] - covid_data['recovered']

covid_data = covid_data.sort_values(by=['country', 'date'])
covid_data['daily_confirmed'] = covid_data.groupby('country')['confirmed'].diff()
covid_data['daily_deaths'] = covid_data.groupby('country')['deaths'].diff()
covid_data['daily_recovered'] = covid_data.groupby('country')['recovered'].diff()

vaccinations_data['date'] = pd.to_datetime(vaccinations_data['date'])

covid_data

In [ ]:
"""Задание 3.1
За какой период представлены данные в таблице covid_data? В качестве ответа введите даты в формате datetime (без указания времени).
Введите ответ в следующем формате (без пробелов): ГГГГ-ММ-ДД-ГГГГ-ММ-ДД"""

print(covid_data['date'].dt.date.min())
print(covid_data['date'].dt.date.max())

In [ ]:
"""Задание 3.2
За какой период представлены данные в таблице vaccinations_data? В качестве ответа введите даты в формате datetime без указания времени.
Введите ответ в следующем формате (без пробелов): ГГГГ-ММ-ДД-ГГГГ-ММ-ДД"""

print(vaccinations_data['date'].dt.date.min())
print(vaccinations_data['date'].dt.date.max())

In [ ]:
"""Задание 3.3
С помощью метода merge() объедините таблицы covid_data и vaccinations_data по столбцам date и country.
Тип объединения выставьте так, чтобы в результирующую таблицу попали только наблюдения за период, вычисленный в задании 3.1. 
То есть в результирующую таблицу должны попасть все записи из таблицы covid_data и из её пересечения с vaccinations_data, но не более. 
Результат объединения занесите в переменную covid_df."""

covid_df = covid_data.merge(
    vaccinations_data, 
    on=['date','country'],
    how='left', 
)

covid_df.to_csv('data/Visualization/covid_df.csv', index=False)
print(covid_df.shape)

covid_df


In [ ]:
"""  Задание 3.4
Какова максимальная летальность в США (United States) за весь период? Ответ округлите до второго знака после запятой."""

import pandas as pd

# 1. Берем только США
usa = covid_df[covid_df['country'] == 'United States'].copy()

# 2. Считаем летальность для КАЖДОГО дня (накопительную)
usa['lethality'] = (usa['deaths'] / usa['confirmed']) * 100

# 3. Находим максимум
max_val = usa['lethality'].max()

print("Максимальная летальность:", round(max_val, 2))

In [ ]:
"""Задание 3.5
Чему равен средний процент выздоровевших в России (Russia)? Ответ округлите до второго знака после запятой."""


rus = covid_df[covid_df['country'] == 'Russia'].copy()


perc = rus['recovered'] / rus['confirmed'] *100

round(perc.mean(), 2)



### За визуализацию графиков в Matplotlib отвечает модуль pyplot

In [ ]:
grouped_cases = covid_df.groupby('date')['daily_confirmed'].sum()


# grouped_cases.plot(
#     kind='line',
#     figsize=(24, 4),
#     title='Ежедневная заболеваемость во времени',
#     grid = True,
#     lw=3
# )

# grouped_cases.plot(
#     kind='hist',
#     figsize=(10, 6),
#     title='Распределение ежедневной заболеваемости',
#     grid = True,
#     color = 'black',
#     bins=10
# );

covid_df.groupby(['country'])['total_vaccinations'].last().nsmallest(5).plot(kind='bar')

# grouped_country.plot(
#     kind='bar',
#     grid=True,
#     figsize=(12, 4),
#     colormap='plasma'
# )

# grouped_country = covid_df.groupby(['country'])[['confirmed', 'deaths']].last()

# grouped_country
# grouped_country = grouped_country.nlargest(10, columns=['confirmed'])
# grouped_country.plot(     kind='bar',     grid=True,     figsize=(12, 4), );

### За визуализацию графиков в Matplotlib отвечает модуль pyplot

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
fig = plt.figure(figsize=(8, 4))
# axes = fig.add_axes([0, 0, 1, 1])

us_data = covid_df[covid_df['country'] == 'United States']

fig = plt.figure(figsize=(6, 3))
# axes = fig.add_axes([0, 0, 1, 1])

# axes = fig.add_axes([0.5, 0.5, 0.5, 0.5])


# axes.scatter(
#     x=us_data['people_fully_vaccinated'], 
#     y=us_data['daily_confirmed'], 
#     s=100,
#     marker='^',
#     c = 'red'
# )

vaccine_combinations = covid_df['vaccines'].value_counts()[:10]
fig = plt.figure(figsize=(5, 5))
axes = fig.add_axes([0, 0, 1, 1])
axes.pie(
    vaccine_combinations,
    labels=vaccine_combinations.index,
    autopct='%.1f%%',
    explode = [0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
);

### Добавление информативности в графики

In [ ]:
china_data = covid_df[covid_df['country'] == 'China']
china_grouped = china_data.groupby(['date'])[['confirmed', 'active', 'deaths', 'recovered']].sum()

#визуализация графиков
fig = plt.figure(figsize=(10, 4))
axes = fig.add_axes([0, 0, 1, 1])
axes.plot(china_grouped['confirmed'], label='Общее число зафиксированных случаев', lw= 3)
axes.plot(china_grouped['deaths'], label='Общее число смертей', lw=3)
axes.plot(china_grouped['recovered'], label='Общее число выздоровевших пациентов', lw=3)
axes.plot(china_grouped['active'], label='Общее число активных случаев', lw=3, linestyle='dashed')



#установка параметров отображения
axes.set_title('Статистика Covid-19 в Китае', fontsize=16)
axes.set_xlabel('Даты')
axes.set_ylabel('Число случаев')
axes.set_yticks(range(0, 100000, 10000))
axes.xaxis.set_tick_params(rotation=30)
axes.grid()
axes.legend();

In [ ]:
vacc_country = covid_df.groupby('country')['people_fully_vaccinated'].last().nlargest(5)
vacc_country_per_hundred = covid_df.groupby('country')['people_fully_vaccinated_per_hundred'].last().nlargest(5)

#визуализация главного графика
fig = plt.figure(figsize=(13, 4))
main_axes = fig.add_axes([0, 0, 1, 1])
main_axes.bar(x = vacc_country.index, height = vacc_country);
main_axes.set_ylabel('Число вакцинированных (2 компонент)')
main_axes.set_title('Топ 5 стран по числу полностью привитых людей')

#визуализация вспомогательного графика
insert_axes = fig.add_axes([0.6, 0.6, 0.38, 0.38])
insert_axes.bar(x = vacc_country_per_hundred.index, height = vacc_country_per_hundred, width=0.5);
insert_axes.set_ylabel('На 100 человек')
insert_axes.xaxis.set_tick_params(rotation=45)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(10, 5))

In [ ]:
import seaborn as sns
print(sns.__version__)

### Seaborn — надстройка над библиотекой Matplotlib, которая значительно расширяет её возможности, позволяя создавать графики более высокого уровня с эстетичным оформлением.

In [ ]:
countries = ['Russia', 'Australia', 'Germany', 'Canada', 'United Kingdom']
croped_covid_df = covid_df[covid_df['country'].isin(countries)]

populations = pd.DataFrame([
    ['Canada', 37664517],
    ['Germany', 83721496],
    ['Russia', 145975300],
    ['Australia', 25726900],
    ['United Kingdom', 67802690]
    ],
    columns=['country', 'population']
)
croped_covid_df = croped_covid_df.merge(populations, on=['country'])
croped_covid_df['daily_confirmed_per_hundred'] = croped_covid_df['daily_confirmed'] / croped_covid_df['population'] * 100
croped_covid_df.head()

### Seaborn метод histplot()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))
sns.histplot(
    data=croped_covid_df,
    x='daily_confirmed_per_hundred',
    bins=25,
    kde=True,
    ax=axes[0]
);
axes[0].set_title('Гистограмма ежедневной заболеваемости на 100 человек', fontsize=16)
sns.histplot(
    data=croped_covid_df,
    x='daily_confirmed_per_hundred',
    y='country',
    bins=25,
    color='red',
    ax=axes[1]
);

### Коробчатые диаграммы в Seaborn строятся с помощью метода boxplot()

In [ ]:
fig = plt.figure(figsize=(10, 7))
boxplot = sns.boxplot(
    data=croped_covid_df,
    y='country',
    x='death_rate',
    orient='h',
    width=0.9
)
boxplot.set_title('Распределение летальности по странам');
boxplot.set_xlabel('Летальность');
boxplot.set_ylabel('Страна');
boxplot.grid()

### Для построения столбчатых диаграмм в Seaborn используется метод barplot()

In [ ]:
fig = plt.figure(figsize=(10, 7))
croped_covid_df['quarter'] = croped_covid_df['date'].dt.quarter
barplot = sns.barplot(
    data=croped_covid_df,
    x='country',
    y='daily_confirmed_per_hundred',
    hue='quarter',
    palette='pastel'
)
barplot.set_title('Средний процент болеющего населения по кварталам');

### jointplot() — в котором совмещены диаграмма рассеяния и гистограмма.

In [ ]:
jointplot = sns.jointplot(
    data=croped_covid_df, 
    x='people_fully_vaccinated_per_hundred', 
    y='daily_confirmed_per_hundred',
    hue='country',
    xlim = (0, 40),
    ylim = (0, 0.1),
    height=8,
    
)

### Тепловая карта  heatmap()

In [ ]:
croped_covid_df['confirmed_per_hundred'] = croped_covid_df['confirmed']/croped_covid_df['population']*100

pivot_1 = croped_covid_df.pivot_table(
    values='confirmed_per_hundred',
    columns='date',
    index='country',
)
pivot_1.columns = pivot_1.columns.astype('string')
display(pivot_1)

heatmap = sns.heatmap(data=pivot_1, cmap='YlGnBu', xticklabels=40)
heatmap.set_title('Тепловая карта вакцинации', fontsize=16)
plt.xticks(rotation=45)
plt.show()


In [ ]:
    croped_covid_df['recover_rate'] = croped_covid_df['recovered']/croped_covid_df['confirmed']*100

    fig = plt.figure(figsize=(10, 7))
    boxplot = sns.boxplot(
        data=croped_covid_df,
        y='country',
        x='recover_rate',
        orient='h',
        width=0.9,
        palette='Set2'
    )
    boxplot.set_title('Oтношение выздоровлений к числу зафиксированных случаев заболевания в процентах по странам');
    boxplot.set_xlabel('recover_rate');
    boxplot.set_ylabel('Страна');
    boxplot.grid()